(multiple-formula-comparison)=
# Multiple Formula Model Comparison

### model_utils.<br>**run_multiple_formula_models**

```python
run_multiple_formula_models(create_model_func, y_val_columns, df_train, formulas, 
                            x_vals=['age', 'sex', 'site'], main_folder=None, save_log=True, 
                            num_cores=None, force_overwrite=False, save_models=False, 
                            use_multiprocessing=True, formula_names=None, eval_data=None, 
                            affected=None, func_name=None)
```

&nbsp;&nbsp;&nbsp;&nbsp;Compares multiple GAMLSS model formulas across different y-values with optional model evaluation. Supports parallel processing for efficiency and caches results to avoid recomputation.

&nbsp;&nbsp;&nbsp;&nbsp;<span style="background-color: #f5f5f5; padding: 4px 8px; display: inline-block; width: 90%;">**Parameters:**</span>

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**create_model_func** : ***callable*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Function to create a GAMLSS model

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**y_val_columns** : ***list of str*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;List of y-value column names to model (one outcome may also be given as a string, e.g. `'thickness'`). All outcomes and formulas are checked before anything is fitted, and one error lists every problem found

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**df_train** : ***pandas.DataFrame*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Training dataframe

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**formulas** : ***list of dict*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;List of formula dictionaries (each with 'mu_formula', 'sigma_formula', 'nu_formula', 'tau_formula')

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**x_vals** : ***list of str***, *default=['age', 'sex', 'site']* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;List of predictor variables

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**main_folder** : ***str***, *default='results'* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Folder for saving results and logs

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**save_log** : ***bool***, *default=True* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Whether to save logs

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**num_cores** : ***int***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Number of CPU cores for multiprocessing (uses 4 if None)

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**force_overwrite** : ***bool***, *default=False* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Whether to overwrite existing cached results. If False, a result saved earlier in main_folder is reused only when it was made with the same settings (formulas, model code, data, number of permutations, evaluation data); results made with other settings are computed again, with a warning. The settings are saved next to each result in a small `_settings.json` file

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**save_models** : ***bool***, *default=False* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Whether to save fitted models

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**use_multiprocessing** : ***bool***, *default=True* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Whether to use multiprocessing

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**formula_names** : ***list of str***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Names for the formulas (defaults to "formula1", "formula2", etc.)

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**eval_data** : ***pandas.DataFrame***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;DataFrame for evaluation

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**affected** : ***array-like***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Array indicating affected status for evaluation. If None, assumes all subjects are healthy. It must have one value per row of eval_data (checked before anything is fitted)

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**func_name** : ***str***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Distribution family name for evaluation (auto-detected if None)

&nbsp;&nbsp;&nbsp;&nbsp;<span style="background-color: #f5f5f5; padding: 4px 8px; display: inline-block; width: 90%;">**Returns:**</span>

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**comparison_df** : ***pandas.DataFrame*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;AIC and BIC values for each formula and y-value

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**convergence_df** : ***pandas.DataFrame*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Convergence information for each formula and y-value

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**summary_df** : ***pandas.DataFrame*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Best models by AIC and BIC for each y-value

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**eval_results** : ***dict***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Evaluation metrics (returned if eval_data provided)

<br>


**Examples:**

In [41]:
import multiprocessing as mp
import os

from normgamlss.gamlss_main import Gamlss
from normgamlss.model_utils import run_multiple_formula_models

# This function gives us the flexibility on what we want to change and what we want to keep constant
# For example, we can keep the nu and tau formula fixed by removing it from this creat_model_func and in the next cell we can
# just change the other parameters or even change the way we want to input

def create_model_func(model_name, x_vals, y_val, mu_formula, sigma_formula, nu_formula, tau_formula):
    def model_func(data):
        gamlss_model = Gamlss(model_name, x_vals, y_val)
        gamlss_model.fit(
            r_code=f"""
            model <- gamlss(
                y ~ {mu_formula},
                sigma.formula = ~ {sigma_formula},
                nu.formula = ~ {nu_formula},
                tau.formula = ~ {tau_formula},
                family = SHASH(),
                method = mixed(10, 100)
            )
            """,
            data=data
        )
        return gamlss_model
    return model_func


In [42]:
train_data1.columns

Index(['site', 'sex', 'age', 'feature_volume1', 'feature_volume2',
       'feature_volume3', 'feature_volume4', 'feature_volume5', 'y_thickness1',
       'y_thickness2', 'y_thickness3', 'y_thickness4'],
      dtype='object')

In [43]:
#### Preparing the input
main_folder = r"../../results/formula_comparison"
num_cores = mp.cpu_count()  # Get the number of CPU cores
y_val_columns = ['y_thickness1','y_thickness2', 'y_thickness3', 'y_thickness4']

# Formula's for baseline and target
formula1 = {'mu_formula': 'pvc(age, by = as.factor(sex)) + re(random=~1|site)',
                'sigma_formula': 'pvc(age) + re(random=~1|site)',
                'nu_formula': '1',
                'tau_formula': '1'}

formula2 = {'mu_formula': 'pvc(age, by = as.factor(sex)) + re(random=~1|site) + feature_volume1',
                'sigma_formula': 'pvc(age) + re(random=~1|site) + feature_volume1 + feature_volume2',
                'nu_formula': '1',
                'tau_formula': '1'}

formula3 = {'mu_formula': 'age + re(random=~1|site)',
                'sigma_formula': '1',
                'nu_formula': '1',
                'tau_formula': '1'}

formula4 = {'mu_formula': 'age',
                'sigma_formula': '1',
                'nu_formula': '1',
                'tau_formula': '1'}

print(f"Number of CPU cores: {num_cores}")


Number of CPU cores: 28


In [44]:
comparison_df, convergence_df, summary_df, eval_results = run_multiple_formula_models(
    create_model_func=create_model_func,
    y_val_columns=y_val_columns,
    df_train=train_data1,
    eval_data=test_data1,
    affected = test_data1['affected'],
    formulas=[formula1, formula2, formula3, formula4],
    x_vals=['age', 'sex', 'site', 'feature_volume1', 'feature_volume2'],
    main_folder=main_folder,
    save_log=True,
    num_cores=4,
    force_overwrite=True,
    save_models=True,
    use_multiprocessing=True
)

2026-10-29 18:08:49,972 - INFO - Using multiprocessing for processing
2026-10-29 18:08:49,974 - INFO - Using 4 cores for multiprocessing


  0%|          | 0/4 [00:00<?, ?it/s]

2026-10-29 18:08:50,193 - INFO - Processing y_thickness1
2026-10-29 18:08:50,195 - INFO - Processing y_thickness2
2026-10-29 18:08:50,198 - INFO - Processing y_thickness4
2026-10-29 18:08:50,198 - INFO - Processing y_thickness3
GAMLSS-RS iteration 1: Global Deviance = 4164.66 
GAMLSS-RS iteration 1: Global Deviance = 3594.915 
GAMLSS-RS iteration 2: Global Deviance = 4119.447 
GAMLSS-RS iteration 1: Global Deviance = 3517.461 
GAMLSS-RS iteration 1: Global Deviance = 3517.72 
GAMLSS-RS iteration 2: Global Deviance = 3509.95 
GAMLSS-RS iteration 3: Global Deviance = 4085.34 
GAMLSS-RS iteration 4: Global Deviance = 4057.893 
GAMLSS-RS iteration 5: Global Deviance = 4037.394 
GAMLSS-RS iteration 2: Global Deviance = 3451.642 
GAMLSS-RS iteration 6: Global Deviance = 4021.056 
GAMLSS-RS iteration 2: Global Deviance = 3438.841 
GAMLSS-RS iteration 3: Global Deviance = 3446.16 
GAMLSS-RS iteration 7: Global Deviance = 4007.78 
GAMLSS-RS iteration 8: Global Deviance = 3997.277 
GAMLSS-RS ite

 25%|██▌       | 1/4 [06:47<20:22, 407.49s/it]

GAMLSS-CG iteration 71: Global Deviance = 3673.505 
GAMLSS-CG iteration 63: Global Deviance = 3061.548 
GAMLSS-CG iteration 72: Global Deviance = 3644.023 
GAMLSS-CG iteration 64: Global Deviance = 3061.55 
GAMLSS-CG iteration 65: Global Deviance = 3061.555 
GAMLSS-CG iteration 66: Global Deviance = 3061.558 
GAMLSS-CG iteration 73: Global Deviance = 3618.227 
GAMLSS-CG iteration 67: Global Deviance = 3061.559 
Model saved to ../../results/formula_comparison/model_cache/saved_models/model_y_thickness2_formula1.rds
Using 370 healthy subjects out of 470 total subjects for evaluation
Using 370 healthy subjects out of 470 total subjects for evaluation
2026-10-29 18:15:56,922 - INFO - Successfully evaluated formula1 for y_thickness2
GAMLSS-CG iteration 74: Global Deviance = 3597.511 
GAMLSS-CG iteration 75: Global Deviance = 3552.104 
GAMLSS-RS iteration 1: Global Deviance = 3516.928 
GAMLSS-CG iteration 76: Global Deviance = 3521.728 
GAMLSS-RS iteration 2: Global Deviance = 3448.371 
GAML

 50%|█████     | 2/4 [16:46<17:19, 519.88s/it]

GAMLSS-CG iteration 97: Global Deviance = 3573.437 
GAMLSS-CG iteration 98: Global Deviance = 3558.089 
GAMLSS-CG iteration 99: Global Deviance = 3549.233 
GAMLSS-CG iteration 100: Global Deviance = 3532.472 
Model saved to ../../results/formula_comparison/model_cache/saved_models/model_y_thickness4_formula2.rds
Using 370 healthy subjects out of 470 total subjects for evaluation
Using 370 healthy subjects out of 470 total subjects for evaluation
2026-10-29 18:25:50,326 - INFO - Successfully evaluated formula2 for y_thickness4
GAMLSS-RS iteration 1: Global Deviance = 3627.222 
GAMLSS-RS iteration 2: Global Deviance = 3542.042 
GAMLSS-RS iteration 3: Global Deviance = 3481.237 
GAMLSS-RS iteration 4: Global Deviance = 3436.7 
GAMLSS-RS iteration 5: Global Deviance = 3402.992 
GAMLSS-RS iteration 6: Global Deviance = 3376.51 
GAMLSS-RS iteration 7: Global Deviance = 3355.555 
GAMLSS-RS iteration 8: Global Deviance = 3338.286 
GAMLSS-RS iteration 9: Global Deviance = 3323.835 
GAMLSS-RS it

100%|██████████| 4/4 [17:20<00:00, 260.14s/it]

2026-10-29 18:26:10,905 - INFO - Processing evaluation results...
2026-10-29 18:26:10,946 - INFO - Evaluation results saved to ../../results/formula_comparison/evaluation_results


In [45]:
comparison_df

,y_val,formula1_AIC,formula1_BIC,formula2_AIC,formula2_BIC,formula3_AIC,formula3_BIC,formula4_AIC,formula4_BIC,best_formula_AIC,best_formula_BIC
0,y_thickness1,3065.477334,3119.575818,-923.125628,-818.508863,3179.910537,3199.659797,3181.923009,3206.609579,formula2 (-923.13),formula2 (-818.51)
1,y_thickness2,3087.693830,3152.212469,8604.764373,8974.744716,3168.616366,3188.365629,3171.012819,3195.699390,formula1 (3087.69),formula1 (3152.21)
2,y_thickness3,3942.367646,3989.406086,3944.960495,4006.470238,3964.706568,3984.455862,3966.755264,3991.441834,formula1 (3942.37),formula3 (3984.46)
3,y_thickness4,3094.552055,3144.257964,3590.670584,3734.343305,3191.867357,3211.616636,3193.816725,3218.503295,formula1 (3094.55),formula1 (3144.26)


In [46]:
summary_df

,y_val,best_AIC_formula,best_AIC_value,best_BIC_formula,best_BIC_value
0,y_thickness1,formula2,-923.125628,formula2,-818.508863
1,y_thickness2,formula1,3087.693830,formula1,3152.212469
2,y_thickness3,formula1,3942.367646,formula3,3984.455862
3,y_thickness4,formula1,3094.552055,formula1,3144.257964


In [47]:
convergence_df

,y_val,formula1_convergence,formula2_convergence,formula3_convergence,formula4_convergence
0,y_thickness1,True,True,True,True
1,y_thickness2,True,False,True,True
2,y_thickness3,True,True,True,True
3,y_thickness4,False,False,True,True


In [48]:
eval_results.keys()

dict_keys(['y_thickness1', 'y_thickness2', 'y_thickness3', 'y_thickness4'])

In [49]:
eval_results['y_thickness1']

,Category,formula1,formula2,formula3,formula4
Metric,,,,,
Z_Scores_Mean,GAMLSS,0.038786,0.051149,0.063466,0.063457
Z_Scores_SD,GAMLSS,0.995145,0.970140,0.982687,0.982684
Logarithmic_Score,GAMLSS,-1.462897,0.469113,-1.527779,-1.527781
Skewness,GAMLSS,-0.019956,0.172875,0.145891,0.145825
Kurtosis,GAMLSS,-0.520425,-0.171350,0.120229,0.119954
Normality_Test_Statistic,GAMLSS,0.992989,0.995408,0.996300,0.996302
MAE,ML,0.943362,0.118975,0.960946,0.960946
MSE,ML,1.225695,0.021875,1.318860,1.318860
RMSE,ML,1.107111,0.147901,1.148416,1.148416


### Loading Models from Batch Operations

When using batch operations like `run_multiple_formula_models`, `model_comparison`, or `run_models_with_permutation`, models are automatically saved in organized directory structures.

**Model Naming Convention:**
- **Formula comparison**: `"model_{y_val}_{formula_name}.rds"`
- **Base/Target comparison**: `"model_{y_val}_{model_type}.rds"` where model_type is 'base' or 'target'
- **Permutation testing**: `"model_{y_val}_{model_type}_perm_{seed}.rds"` for permuted models

**Note**: The models are saved in `{main_folder}/model_cache/saved_models/`

**Examples:**

In [50]:
## Example code to load a saved model

# Update these values to match your setup
main_folder = r"../../results/formula_comparison"  # Path to your main results folder
y_val = "y_thickness2"  # Replace with your outcome variable
formula_name = "formula1"  # String for the formula number

# The rest is derived automatically from the values above
model_name = f"model_{y_val}_{formula_name}"
model_path = os.path.join(main_folder, "model_cache", "saved_models", f"{model_name}.rds")
loaded_model = Gamlss.load_model(model_path)

Model 'model_y_thickness2_formula1' loaded from ../../results/formula_comparison/model_cache/saved_models/model_y_thickness2_formula1.rds


In [51]:
# The loaded model can now be used like any other -- for example, to calculate z-scores
z_scores = loaded_model.z_score(train_data1,train_data1, func_name = 'SHASH')
print('Z-score:', z_scores)

Z-score: [ 0.158061   -1.00684254  1.34560406 ... -0.66780727 -0.22300694
  1.47957539]
